# Actual Working with Arena Data

In [2]:
from datasets import load_dataset

ds = load_dataset("lmarena-ai/arena-human-preference-140k")

In [3]:
ds_dict = dict(ds)['train']

In [4]:
ds_dict['category_tag'][0:5]

[{'creative_writing_v0.1': {'creative_writing': False, 'score': 'no'},
  'criteria_v0.1': {'complexity': False,
   'creativity': False,
   'domain_knowledge': True,
   'problem_solving': True,
   'real_world': False,
   'specificity': False,
   'technical_accuracy': True},
  'if_v0.1': {'if': False, 'score': 1},
  'math_v0.1': {'math': False}},
 {'creative_writing_v0.1': {'creative_writing': False, 'score': 'no'},
  'criteria_v0.1': {'complexity': True,
   'creativity': True,
   'domain_knowledge': True,
   'problem_solving': True,
   'real_world': True,
   'specificity': True,
   'technical_accuracy': True},
  'if_v0.1': {'if': True, 'score': 4},
  'math_v0.1': {'math': False}},
 {'creative_writing_v0.1': {'creative_writing': False, 'score': 'no'},
  'criteria_v0.1': {'complexity': True,
   'creativity': False,
   'domain_knowledge': True,
   'problem_solving': True,
   'real_world': True,
   'specificity': True,
   'technical_accuracy': True},
  'if_v0.1': {'if': False, 'score': 3},


In [5]:
_ = [print(k) for k in ds_dict.features.keys()]

id
model_a
model_b
winner
evaluation_session_id
evaluation_order
conversation_a
conversation_b
full_conversation
conv_metadata
category_tag
language
is_code
timestamp


In [6]:
ds_dict['conv_metadata'][0]

{'sum_assistant_a_tokens': 1854,
 'header_count_a': {'h1': 0, 'h2': 0, 'h3': 8, 'h4': 0, 'h5': 0, 'h6': 0},
 'list_count_a': {'ordered': 0, 'unordered': 39},
 'bold_count_a': {'**': 65, '__': 0},
 'context_a_tokens': 11,
 'sum_assistant_b_tokens': 264,
 'header_count_b': {'h1': 1, 'h2': 0, 'h3': 0, 'h4': 0, 'h5': 0, 'h6': 0},
 'list_count_b': {'ordered': 0, 'unordered': 0},
 'bold_count_b': {'**': 5, '__': 0},
 'context_b_tokens': 11,
 'sum_user_tokens': 11,
 'turns': 1}

In [7]:
ds_dict[0]

{'id': 'c4b9710c-8d64-4bee-a0b0-94637ae4cc65',
 'model_a': 'gemini-2.5-pro',
 'model_b': 'claude-3-7-sonnet-20250219-thinking-32k',
 'winner': 'model_a',
 'evaluation_session_id': 'a333a685-37f9-474d-b703-f079d8329552',
 'evaluation_order': 1,
 'conversation_a': [{'role': 'user',
   'content': [{'type': 'text',
     'text': 'Compare Tormenta20 with DnD5e',
     'image': None,
     'mimeType': None}]},
  {'role': 'assistant',
   'content': [{'type': 'text',
     'text': 'Excellent question! Comparing **Tormenta20 (T20)** and **Dungeons & Dragons 5th Edition (D&D 5e)** is a fantastic way to understand different approaches to the d20 system. While they share a common ancestor (D&D 3rd Edition), they have evolved in vastly different directions.\n\nHere is a detailed comparison, broken down by key aspects of the games.\n\n---\n\n### **High-Level Philosophy**\n\n*   **D&D 5e:** Aims for **accessibility, streamlined play, and heroic fantasy**. It simplifies many of the complex rules of older 

In [8]:
k = set()
for i in range(5):
    k.update(ds_dict['category_tag'][i].keys())

In [9]:
k

{'creative_writing_v0.1', 'criteria_v0.1', 'if_v0.1', 'math_v0.1'}

## Examining Only Coding Problems

### Initial Filtering

In [10]:
idxs = ds_dict['is_code']

In [12]:
ds_dict[idxs[:50]]['conversation_a']

[[{'role': 'user',
   'content': [{'type': 'text',
     'text': 'Compare Tormenta20 with DnD5e',
     'image': None,
     'mimeType': None}]},
  {'role': 'assistant',
   'content': [{'type': 'text',
     'text': 'Excellent question! Comparing **Tormenta20 (T20)** and **Dungeons & Dragons 5th Edition (D&D 5e)** is a fantastic way to understand different approaches to the d20 system. While they share a common ancestor (D&D 3rd Edition), they have evolved in vastly different directions.\n\nHere is a detailed comparison, broken down by key aspects of the games.\n\n---\n\n### **High-Level Philosophy**\n\n*   **D&D 5e:** Aims for **accessibility, streamlined play, and heroic fantasy**. It simplifies many of the complex rules of older editions to be welcoming to new players. Its design philosophy is "rulings, not rules," encouraging Dungeon Masters to make calls on the fly. It is built to be setting-agnostic, though the Forgotten Realms is its default.\n*   **Tormenta20:** Aims for **deep cha

In [18]:
import numpy as np

In [19]:
en_idxs = (np.array(ds_dict['language']) == 'en')

In [20]:
en_idxs

array([ True, False,  True, ..., False,  True,  True])

In [23]:
code_idxs = ds_dict['is_code']

In [24]:
final_idxs = en_idxs & code_idxs

In [25]:
final_idxs

array([False, False,  True, ..., False,  True, False])

In [26]:
final_ds = ds_dict[final_idxs]

In [32]:
final_idxs.sum()

24366

In [28]:
en_idxs.sum()

71175

In [30]:
np.array(code_idxs).sum()

39363

In [33]:
model_as = set(final_ds['model_a'])
model_bs = set(final_ds['model_b'])
candidates = model_as.union(model_bs)

In [42]:
candidates_idxs = {candidates : i for i, candidates in enumerate(candidates)}

In [43]:
candidates_idxs

{'gemini-2.5-pro': 0,
 'claude-3-5-haiku-20241022': 1,
 'claude-3-7-sonnet-20250219-thinking-32k': 2,
 'claude-3-5-sonnet-20241022': 3}

Might be a bit out of date, since there are only four candidates? Will be good for proof of concept.

In [39]:
final_ds['model_a'][:10]

['gemini-2.5-pro',
 'gemini-2.5-pro',
 'claude-3-5-haiku-20241022',
 'gemini-2.5-pro',
 'gemini-2.5-pro',
 'gemini-2.5-pro',
 'gemini-2.5-pro',
 'gemini-2.5-pro',
 'gemini-2.5-pro',
 'claude-3-5-haiku-20241022']

In [ ]:
final_ds.keys

dict_keys(['id', 'model_a', 'model_b', 'winner', 'evaluation_session_id', 'evaluation_order', 'conversation_a', 'conversation_b', 'full_conversation', 'conv_metadata', 'category_tag', 'language', 'is_code', 'timestamp'])

In [56]:
winners = []
losers = []
n_items = len(candidates)

In [57]:
for i in range(len(final_ds['id'])):
    if final_ds['winner'][i] == 'model_a':
        winner = candidates_idxs[final_ds['model_a'][i]]
        loser = candidates_idxs[final_ds['model_b'][i]]
        
    elif final_ds['winner'][i] == 'model_b':
        winner = candidates_idxs[final_ds['model_b'][i]]
        loser = candidates_idxs[final_ds['model_a'][i]]

    winners.append(winner)
    losers.append(loser)

In [58]:
print(len(winners))
print(len(losers))

135634
135634


In [59]:
from scipy.optimize import minimize
from scipy.special import expit

In [60]:
r_free = np.zeros(n_items - 1)

In [96]:
def bt_neg_log_likelihood(r_free, winners, losers, n_items):
    """
    r_free: parameters for items 0..n_items-2
    last item's reward is fixed to 0 for identifiability
    winners, losers: arrays of item indices for each comparison
    """
    r = np.concatenate([r_free, [0.0]])  # fix last reward = 0

    diff = r[winners] - r[losers]
    
    # log sigma(diff) = -log(1 + exp(-diff))
    # use stable form:
    nll = np.sum(np.logaddexp(0.0, -diff))
    return nll

def fit_bradley_terry(winners, losers, n_items):
    """
    winners[k] beat losers[k]
    """
    x0 = np.zeros(n_items - 1)

    result = minimize(
        bt_neg_log_likelihood,
        x0,
        args=(winners, losers, n_items),
        method="L-BFGS-B"
    )

    r_hat = np.concatenate([result.x, [0.0]])
    return r_hat, result

In [62]:
r_hat, result = fit_bradley_terry(winners, losers, n_items)

In [63]:
r_hat

array([ 11.9562009 ,   0.        , -12.00242633,   0.        ])

In [64]:
result

  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 5.336653086431478e-06
        x: [ 1.196e+01  0.000e+00 -1.200e+01]
      nit: 33
      jac: [-5.337e-06  0.000e+00  5.337e-06]
 hess_inv: [[ 1.617e+04  0.000e+00 -1.623e+04]
            [ 0.000e+00  1.000e+00  0.000e+00]
            [-1.623e+04  0.000e+00  1.630e+04]]
     nfev: 136
     njev: 34

Seems like this could be reasonable, on https://arena.ai/leaderboard/code, only gemini-2.5-pro makes the cut.

Need to find a category where there is enough overlap in the models considered.

## Modeling All the Data Distributions

In [65]:
ds_dict

Dataset({
    features: ['id', 'model_a', 'model_b', 'winner', 'evaluation_session_id', 'evaluation_order', 'conversation_a', 'conversation_b', 'full_conversation', 'conv_metadata', 'category_tag', 'language', 'is_code', 'timestamp'],
    num_rows: 135634
})

In [66]:
all_candidates = set(ds_dict['model_a']).union(set(ds_dict['model_b']))

In [68]:
all_candidate_idxs = {cand : i for i, cand in enumerate(all_candidates)}
print(all_candidate_idxs)

{'claude-opus-4-20250514': 0, 'gemini-2.5-pro': 1, 'gpt-4.1-2025-04-14': 2, 'mistral-small-3.1-24b-instruct-2503': 3, 'gemini-2.5-flash-lite-preview-06-17-thinking': 4, 'gemini-2.5-flash-preview-04-17': 5, 'llama-4-maverick-03-26-experimental': 6, 'kimi-k2-0711-preview': 7, 'claude-sonnet-4-20250514': 8, 'claude-3-7-sonnet-20250219-thinking-32k': 9, 'grok-4-0709': 10, 'gpt-4o-mini-2024-07-18': 11, 'qwq-32b': 12, 'grok-3-preview-02-24': 13, 'qwen3-235b-a22b': 14, 'gpt-4.1-mini-2025-04-14': 15, 'magistral-medium-2506': 16, 'mistral-small-2506': 17, 'o4-mini-2025-04-16': 18, 'deepseek-v3-0324': 19, 'claude-3-5-haiku-20241022': 20, 'qwen3-235b-a22b-no-thinking': 21, 'qwen3-235b-a22b-instruct-2507': 22, 'gemini-2.5-flash': 23, 'chatgpt-4o-latest-20250326': 24, 'gemini-2.5-pro-preview-03-25': 25, 'qwen3-30b-a3b': 26, 'gemini-2.0-flash-thinking-exp-01-21': 27, 'gemma-3-27b-it': 28, 'grok-3-mini-beta': 29, 'qwen3-coder-480b-a35b-instruct': 30, 'mistral-medium-2505': 31, 'o3-mini': 32, 'claude-

In [73]:
n_items = len(all_candidate_idxs)

### Math

In [70]:
ds_dict[0]['category_tag']

{'creative_writing_v0.1': {'creative_writing': False, 'score': 'no'},
 'criteria_v0.1': {'complexity': False,
  'creativity': False,
  'domain_knowledge': True,
  'problem_solving': True,
  'real_world': False,
  'specificity': False,
  'technical_accuracy': True},
 'if_v0.1': {'if': False, 'score': 1},
 'math_v0.1': {'math': False}}

In [71]:
ds_dict['category_tag'][0]

{'creative_writing_v0.1': {'creative_writing': False, 'score': 'no'},
 'criteria_v0.1': {'complexity': False,
  'creativity': False,
  'domain_knowledge': True,
  'problem_solving': True,
  'real_world': False,
  'specificity': False,
  'technical_accuracy': True},
 'if_v0.1': {'if': False, 'score': 1},
 'math_v0.1': {'math': False}}

In [72]:
from tqdm import tqdm

math_idxs = []

for i in tqdm(range(ds_dict.num_rows)):
    if ds_dict[i]['category_tag']['math_v0.1']['math']:
        math_idxs.append(i)

100%|██████████| 135634/135634 [00:49<00:00, 2717.44it/s]


In [77]:
winners = []
losers = []

for j in tqdm(range(len(math_idxs))):
    i = math_idxs[j]
    if ds_dict['winner'][i] == 'model_a':
        winner = all_candidate_idxs[ds_dict['model_a'][i]]
        loser = all_candidate_idxs[ds_dict['model_b'][i]]
        
    elif ds_dict['winner'][i] == 'model_b':
        winner = all_candidate_idxs[ds_dict['model_b'][i]]
        loser = all_candidate_idxs[ds_dict['model_a'][i]]

    winners.append(winner)
    losers.append(loser)

100%|██████████| 10892/10892 [01:36<00:00, 112.29it/s]


In [79]:
print(len(winners))
print(len(losers))
print(len(math_idxs))

10892
10892
10892


In [97]:
r_hat, result = fit_bradley_terry(winners, losers, n_items)
print(r_hat)
print(result)

[-0.03130526  1.01294038 -0.40747008 -1.37056255  0.02170627 -0.05116355
  0.32720692 -0.08973024 -0.22254014 -0.69041123  0.85221391 -1.24685424
 -0.37354339 -0.12839918  0.36316186 -0.53425018 -1.30201259 -0.66733382
 -0.04703055 -0.32830716 -1.31001806  0.0995934  -0.31453801  0.42618869
  0.22502245  0.67404478 -0.14456104 -0.66688468 -0.44754513 -0.46457851
  0.         -0.42770678 -0.35336755 -0.64671472  0.13322405 -0.11514429
  0.21052907 -0.92463141  0.05630352 -1.20946801 -0.89995963 -0.1593359
  0.38763454 -0.91715935 -0.99273159  0.1084732  -0.90480738 -1.37114316
 -0.66830492 -0.3986288  -0.14554825 -0.69385394  0.        ]
  message: CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
  success: True
   status: 0
      fun: 6856.527067820896
        x: [-3.131e-02  1.013e+00 ... -1.455e-01 -6.939e-01]
      nit: 26
      jac: [ 4.002e-03 -2.547e-03 ... -5.457e-04  3.638e-04]
     nfev: 1484
     njev: 28
 hess_inv: <52x52 LbfgsInvHessProduct with dtype=float64>


In [98]:
r_hat = np.array(r_hat)

In [99]:
all_candidate_idxs['gemini-2.5-pro']

1

In [100]:
true_ranking = np.argsort(-r_hat)
print(true_ranking)

[ 1 10 25 23 42 14  6 24 36 34 45 21 38  4 52 30  0 18  5  7 35 13 26 50
 41  8 22 19 32 12 49  2 31 28 29 15 33 27 17 48  9 51 40 46 43 37 44 39
 11 16 20  3 47]


In [101]:
def borda_from_pairwise(winners, losers, n_items=None):
    winners = np.asarray(winners)
    losers = np.asarray(losers)

    if n_items is None:
        n_items = max(winners.max(), losers.max()) + 1

    scores = np.zeros(n_items, dtype=int)

    for w in winners:
        scores[w] += 1

    denom = np.zeros(n_items, dtype=int)
    for l in losers:
        denom[l] += 1

    denom = denom + scores + 1e-6
    scores = scores / denom

    ranking = np.argsort(-scores)  # descending
    return scores, ranking

scores, ranking = borda_from_pairwise(winners, losers)
print(ranking)

[ 1 10 25 23 14 42  6 36 24 34 45 21  4  5 52  0 18 38 35  7 26 50 41 13
  8 12 19 32 31  2 49 22 29 27 28 15 17 48 33  9 37 51 40 43 46 11 44 39
 20 47 16  3 30]


In [125]:
def leaderboard_dist(ranking, true_ranking, avg_utils):
    ranking = np.asarray(ranking)
    true_ranking = np.asarray(true_ranking)

    ranking_utils = avg_utils[ranking]
    true_ranking_utils = avg_utils[true_ranking]

    denom_cumsum = np.cumsum(ranking_utils)
    num_cumsum = np.cumsum(true_ranking_utils)

    ratio = np.max(num_cumsum / denom_cumsum)
    k = np.argmax(num_cumsum / denom_cumsum)
    return ratio, k

In [103]:
leaderboard_dist(ranking, true_ranking, r_hat)

25.031109480782472

### Subdividing the Category Further

Seems like there is still distortion when doing math.

In [106]:
ds_dict[0]['language']

'en'

In [107]:
winners = []
losers = []

for i in tqdm(range(ds_dict.num_rows)):

    if ds_dict[i]['category_tag']['math_v0.1']['math'] and ds_dict[i]['language'] == 'en':

        if ds_dict['winner'][i] == 'model_a':
            winner = all_candidate_idxs[ds_dict['model_a'][i]]
            loser = all_candidate_idxs[ds_dict['model_b'][i]]
            
        elif ds_dict['winner'][i] == 'model_b':
            winner = all_candidate_idxs[ds_dict['model_b'][i]]
            loser = all_candidate_idxs[ds_dict['model_a'][i]]

        winners.append(winner)
        losers.append(loser)


print(len(winners))
print(len(losers))

100%|██████████| 135634/135634 [01:39<00:00, 1368.84it/s]

5624
5624


In [108]:
r_hat, result = fit_bradley_terry(winners, losers, n_items)
print(r_hat)
print(result)

[ 9.11353065e-02  8.49187807e-01 -2.46195572e-01 -1.06392375e+00
 -3.05848025e-01  2.65866808e-01  3.42770918e-01  1.49644952e-02
 -2.48779903e-02 -7.82455021e-01  5.02269765e-01 -8.65974022e-01
 -3.70024332e-01  1.32194900e-02  6.14625852e-01 -4.07356297e-01
 -1.48038832e+00 -5.08591540e-01  1.18087241e-03 -1.68491618e-01
 -1.17377707e+00  1.96012199e-01  5.75952290e-02  4.48168205e-01
  3.72891493e-01  9.13008041e-01 -2.08189331e-01  5.85027734e-02
 -4.17513702e-01 -2.27175337e-01  0.00000000e+00 -3.79972007e-01
 -1.91240681e-01 -5.55052770e-01  9.58598972e-02  3.18917457e-01
  3.52021924e-01 -7.22334056e-01 -9.74025502e-02 -9.96763760e-01
 -7.88009978e-01 -5.48634196e-01  1.25473287e+00 -5.08384011e-01
 -1.13548959e+00  3.05900186e-01 -1.01154423e+00 -9.46278435e-01
 -6.43043709e-01 -3.99006493e-01  6.75292001e-03 -9.43430976e-01
  0.00000000e+00]
  message: CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
  success: True
   status: 0
      fun: 3510.5565717912323
        x: [ 9.114e

In [109]:
true_ranking = np.argsort(-r_hat)
print(true_ranking)

[42 25  1 14 10 23 24 36  6 35 45  5 21 34  0 27 22  7 13 50 18 30 52  8
 38 19 32 26 29  2  4 12 31 49 15 28 43 17 41 33 48 37  9 40 11 51 47 39
 46  3 44 20 16]


In [110]:
scores, ranking = borda_from_pairwise(winners, losers)
print(ranking)

[42 25  1 14 23 35 36 10 24  6 45  5 21 27  0 34 18 52 50  7  8 22 13 38
 19 29 26 32  4  2 15 31 12 49 28 43 48 17 37 41 33 40  9 11 47 51 46 39
  3 20 44 16 30]


In [111]:
leaderboard_dist(ranking, true_ranking, r_hat)

5.939026416211874

### Trying the Same Thing, But Code

In [120]:
winners = []
losers = []

for i in tqdm(range(ds_dict.num_rows)):

    row = ds_dict[i]
    if row['is_code'] and row['language'] == 'en':

        w = row['winner']

        if w == 'model_a':
            winners.append(all_candidate_idxs[row['model_a']])
            losers.append(all_candidate_idxs[row['model_b']])
            
        elif w == 'model_b':
            winners.append(all_candidate_idxs[row['model_b']])
            losers.append(all_candidate_idxs[row['model_a']])

print(len(winners))
print(len(losers))

100%|██████████| 135634/135634 [00:46<00:00, 2890.59it/s]

17525
17525


In [121]:
n_items = len(all_candidate_idxs)
r_hat, result = fit_bradley_terry(winners, losers, n_items)
print(r_hat)
print(result)

[ 0.18828495  0.7405148   0.2089593  -0.74979348 -0.05559935  0.03226814
  0.3756478   0.07603939 -0.11710929 -0.42117171  0.50210541 -0.97319273
 -0.10795888  0.46783263  0.18429077  0.06344045 -0.55412223  0.12588312
  0.14174413  0.06618557 -0.79253499  0.16500241  0.17023958  0.35214251
  0.45798973  0.6946004  -0.19269329 -0.27769044 -0.15053831  0.08898393
  0.          0.1296758  -0.25325041 -0.43056355  0.22497234  0.04901509
  0.49422687  0.33938702  0.04538006 -0.84929636 -0.52042769  0.51110641
 -0.04313066 -0.31871241 -0.50168973  0.38930902 -0.5779341  -0.84906995
 -0.39832417 -0.23582471  0.27863763 -0.25727696  0.        ]
  message: CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
  success: True
   status: 0
      fun: 11484.126014781645
        x: [ 1.883e-01  7.405e-01 ...  2.786e-01 -2.573e-01]
      nit: 25
      jac: [-1.837e-02 -1.819e-03 ...  1.273e-03 -7.276e-04]
     nfev: 1431
     njev: 27
 hess_inv: <52x52 LbfgsInvHessProduct with dtype=float64>


In [122]:
true_ranking = np.argsort(-r_hat)
print(true_ranking)

[ 1 25 41 10 36 13 24 45  6 23 37 50 34  2  0 14 22 21 18 31 17 29  7 19
 15 35 38  5 30 52 42  4 12  8 28 26 49 32 51 27 43 48  9 33 44 40 16 46
  3 20 47 39 11]


In [123]:
scores, ranking = borda_from_pairwise(winners, losers)
print(ranking)

[ 1 25 10 36 24 41 37 13 45  6 23 50 34  2 21 22 14  0 18 17 31  7 35 29
 19  5 15 38 52  4 42 12  8 28 27 26 49 32 51 43 48  9 33 40 44 16 46  3
 20 47 39 11 30]


In [126]:
leaderboard_dist(ranking, true_ranking, r_hat)

(2.3045437955506864, 48)

In [131]:
[c for c, v in all_candidate_idxs.items() if v == 41]

['deepseek-r1-0528']

In [133]:
len(set(winners).union(set(losers)))

52